# Spatiotemporal Taxi Demand 2018 (XGBoost)


In [1]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Kill Spark cũ nếu còn sót
import os

os.system("pkill -f 'java.*spark' 2>/dev/null || true")
import time
time.sleep(3)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import numpy as np
import pandas as pd

from sklearn.decomposition import KernelPCA
from sklearn.cluster import KMeans
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from functools import reduce

spark = (
    SparkSession.builder
    .appName("TaxiPipeline_V4_XGBoost")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "32")
    .config("spark.default.parallelism", "32")
    .config("spark.sql.parquet.enableVectorizedReader", "false")
    .getOrCreate()
)

paths = [
    f"/user/data/raw/yellow_tripdata_{year}-{month:02d}.parquet"
    for year in range(2018, 2019)
    for month in range(1, 13)
]

dfs = []
bad_files = []

for p in paths:
    try:
        tmp = spark.read.parquet(p).select(
            F.col("tpep_pickup_datetime").cast("timestamp").alias("tpep_pickup_datetime"),
            F.col("PULocationID").cast("int").alias("PULocationID")
        )
        dfs.append(tmp)
        print("OK:", p)
    except Exception as e:
        bad_files.append((p, str(e)))
        print("FAIL:", p, e)

if not dfs:
    raise ValueError("Không đọc được file nào.")

df = reduce(lambda a, b: a.unionByName(b), dfs)

print("Good files:", len(dfs))
print("Bad files:", len(bad_files))
df.printSchema()
print("Rows:", df.count())

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-00749d32-cb3f-4efe-99b8-41d923607290;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 173ms :: artifacts dl 5ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	-----------------------------------------------

OK: /user/data/raw/yellow_tripdata_2018-01.parquet
OK: /user/data/raw/yellow_tripdata_2018-02.parquet
OK: /user/data/raw/yellow_tripdata_2018-03.parquet
OK: /user/data/raw/yellow_tripdata_2018-04.parquet
OK: /user/data/raw/yellow_tripdata_2018-05.parquet
OK: /user/data/raw/yellow_tripdata_2018-06.parquet
OK: /user/data/raw/yellow_tripdata_2018-07.parquet
OK: /user/data/raw/yellow_tripdata_2018-08.parquet
OK: /user/data/raw/yellow_tripdata_2018-09.parquet
OK: /user/data/raw/yellow_tripdata_2018-10.parquet
OK: /user/data/raw/yellow_tripdata_2018-11.parquet
OK: /user/data/raw/yellow_tripdata_2018-12.parquet
Good files: 12
Bad files: 0
root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)



Rows: 102871387


In [3]:
# Lọc thời gian hợp lệ cho dữ liệu 2018
df_input = df.filter(
    (F.col("tpep_pickup_datetime") >= F.lit("2018-01-01 00:00:00")) &
    (F.col("tpep_pickup_datetime") < F.lit("2019-01-01 00:00:00"))
)

df_input.select(
    F.min("tpep_pickup_datetime").alias("min_time"),
    F.max("tpep_pickup_datetime").alias("max_time")
).show(truncate=False)

+-------------------+-------------------+
|min_time           |max_time           |
+-------------------+-------------------+
|2018-01-01 00:00:00|2018-12-31 23:59:58|
+-------------------+-------------------+



In [4]:
SLOT_MINUTES = 30
MIN_MEAN_PICKUPS_PER_SLOT = 10.0
N_CLUSTERS = 8
CLUSTER_LAG = 12
DISAGG_LAG = 12
TRAIN_RATIO = 0.8
RANDOM_STATE = 42

XGB_CLUSTER_PARAMS = dict(
    n_estimators=100,
    max_depth=12,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    n_jobs=1,
    random_state=RANDOM_STATE,
    verbosity=0
)

XGB_DISAGG_PARAMS = dict(
    n_estimators=50,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=2,
    n_jobs=1,
    random_state=RANDOM_STATE,
    verbosity=0
)

In [5]:
# ============================================================
# HELPER FUNCTIONS — Preprocessing, Clustering, Metrics
# ============================================================

def preprocess_pickup_counts_spark(
    df,
    slot_minutes=30,
    pickup_col="tpep_pickup_datetime",
    zone_col="PULocationID"
):
    slot_seconds = slot_minutes * 60
    sdf = (
        df.select(pickup_col, zone_col)
        .filter(F.col(pickup_col).isNotNull())
        .filter(F.col(zone_col).isNotNull())
        .withColumn(zone_col, F.col(zone_col).cast("int"))
        .filter(F.col(zone_col) > 0)
        .withColumn("pickup_ts", F.col(pickup_col).cast("timestamp"))
        .withColumn(
            "slot_unix",
            (F.floor(F.unix_timestamp("pickup_ts") / slot_seconds) * slot_seconds).cast("long")
        )
        .withColumn("slot_ts", F.to_timestamp(F.from_unixtime(F.col("slot_unix"))))
        .groupBy("slot_ts", zone_col)
        .agg(F.count(F.lit(1)).alias("pickup_count"))
        .withColumnRenamed(zone_col, "zone_id")
        .select("slot_ts", "zone_id", "pickup_count")
    )
    return sdf


def build_dense_zone_slot_matrix_spark(
    pickup_counts_sdf,
    active_zones,
    slot_minutes=30
):
    sdf = pickup_counts_sdf.filter(F.col("zone_id").isin(active_zones))
    min_max = sdf.select(
        F.min("slot_ts").alias("min_ts"),
        F.max("slot_ts").alias("max_ts")
    ).collect()[0]
    min_ts = min_max["min_ts"]
    max_ts = min_max["max_ts"]

    full_slots = spark.sql(f"""
        SELECT explode(
            sequence(
                timestamp('{min_ts}'),
                timestamp('{max_ts}'),
                interval {slot_minutes} minutes
            )
        ) AS slot_ts
    """)

    zones_df = spark.createDataFrame([(z,) for z in active_zones], ["zone_id"])
    dense_base = full_slots.crossJoin(zones_df)
    dense_counts = (
        dense_base
        .join(sdf, on=["slot_ts", "zone_id"], how="left")
        .fillna(0, subset=["pickup_count"])
    )
    pivot_sdf = (
        dense_counts
        .groupBy("slot_ts")
        .pivot("zone_id", active_zones)
        .sum("pickup_count")
        .na.fill(0)
        .orderBy("slot_ts")
    )

    pdf = pivot_sdf.toPandas()
    pdf["slot_ts"] = pd.to_datetime(pdf["slot_ts"])
    pdf = pdf.set_index("slot_ts")
    pdf.columns = [int(c) for c in pdf.columns]
    pdf = pdf.sort_index(axis=1)
    return pdf.astype(np.float32)


def get_active_zones_spark(pickup_counts_sdf, min_mean_pickups_per_slot=10.0):
    n_slots = pickup_counts_sdf.select("slot_ts").distinct().count()
    zone_stats = (
        pickup_counts_sdf
        .groupBy("zone_id")
        .agg(F.sum("pickup_count").alias("total_pickups"))
        .withColumn("mean_pickups_per_slot", F.col("total_pickups") / F.lit(n_slots))
        .filter(F.col("mean_pickups_per_slot") >= F.lit(min_mean_pickups_per_slot))
        .orderBy("zone_id")
    )
    return [r["zone_id"] for r in zone_stats.select("zone_id").collect()]


def compute_correlation_dissimilarity(zone_ts_pdf: pd.DataFrame):
    corr_df = zone_ts_pdf.corr(method="pearson")
    corr_df = corr_df.fillna(0.0).clip(-1.0, 1.0)
    np.fill_diagonal(corr_df.values, 1.0)
    dist_df = 1.0 - corr_df
    return corr_df, dist_df


def gaussian_kernel_on_distance_rows(dist_df: pd.DataFrame):
    D = dist_df.values.astype(np.float64)
    L = D.shape[0]
    row_sq = np.sum(D * D, axis=1, keepdims=True)
    sq_euclidean = row_sq + row_sq.T - 2 * (D @ D.T)
    sq_euclidean = np.maximum(sq_euclidean, 0.0)
    G = np.exp(-sq_euclidean / max(L, 1))
    return pd.DataFrame(G, index=dist_df.index, columns=dist_df.columns)


def cluster_zones_kernel_pca_kmeans(
    dist_df: pd.DataFrame,
    n_clusters: int = 8,
    random_state: int = 42
):
    G_df = gaussian_kernel_on_distance_rows(dist_df)
    n_components = min(max(n_clusters, 2), max(1, G_df.shape[0] - 1))
    kpca = KernelPCA(
        n_components=n_components,
        kernel="precomputed",
        random_state=random_state
    )
    embedding = kpca.fit_transform(G_df.values)
    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=20
    )
    labels = kmeans.fit_predict(embedding)
    cluster_map = pd.Series(labels, index=dist_df.index, name="cluster_id")
    return cluster_map, G_df, embedding


def aggregate_cluster_demand(zone_ts_pdf: pd.DataFrame, cluster_map: pd.Series):
    cluster_ts = {}
    for c in sorted(cluster_map.unique()):
        zones = cluster_map[cluster_map == c].index.tolist()
        cluster_ts[c] = zone_ts_pdf[zones].sum(axis=1)
    cluster_ts_pdf = pd.DataFrame(cluster_ts, index=zone_ts_pdf.index).sort_index(axis=1)
    cluster_ts_pdf.columns = [f"cluster_{c}" for c in cluster_ts_pdf.columns]
    return cluster_ts_pdf


def make_lagged_supervised(series: np.ndarray, lag: int):
    X, y = [], []
    for t in range(lag, len(series)):
        X.append(series[t-lag:t][::-1])
        y.append(series[t])
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)


def chronological_train_test_split(X, y, train_ratio=0.8):
    n = len(X)
    split = int(n * train_ratio)
    return X[:split], X[split:], y[:split], y[split:]


def masked_mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    mask = y_true > 0
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100.0)


def wmape(y_true, y_pred, eps=1e-9):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    denom = np.sum(np.abs(y_true))
    if denom < eps:
        return np.nan
    return float(np.sum(np.abs(y_true - y_pred)) / denom * 100.0)


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    try:
        r2 = float(r2_score(y_true, y_pred))
    except:
        r2 = np.nan
    return {"MAE": mae, "RMSE": rmse, "MAPE": masked_mape(y_true, y_pred),
            "WMAPE": wmape(y_true, y_pred), "R2": r2}


def nanmean_safe(values):
    vals = np.asarray(values, dtype=np.float64)
    if np.all(np.isnan(vals)):
        return np.nan
    return float(np.nanmean(vals))


# ============================================================
# METRICS ĐÚNG THEO PAPER:
# Tính metrics cho TỪNG zone riêng lẻ, rồi average across all zones
# Paper: "the above metrics are averaged across the 57 taxi zones"
# ============================================================

def compute_paper_zone_averaged_metrics(zone_level_predictions):
    """
    Đúng theo paper: tính metrics cho từng zone, rồi average.
    Khác với global pooled (ghép hết rồi tính 1 lần).
    """
    all_zone_mae = []
    all_zone_rmse = []
    all_zone_mape = []
    all_zone_wmape = []
    all_zone_r2 = []

    for c, pred in zone_level_predictions.items():
        Y_test = pred["y_test"]   # shape (T_test, n_zones)
        Y_pred = pred["y_pred"]   # shape (T_test, n_zones)
        zones = pred["zones"]

        for i, z in enumerate(zones):
            m = regression_metrics(Y_test[:, i], Y_pred[:, i])
            all_zone_mae.append(m["MAE"])
            all_zone_rmse.append(m["RMSE"])
            all_zone_mape.append(m["MAPE"])
            all_zone_wmape.append(m["WMAPE"])
            all_zone_r2.append(m["R2"])

    return {
        "MAE": nanmean_safe(all_zone_mae),
        "RMSE": nanmean_safe(all_zone_rmse),
        "MAPE": nanmean_safe(all_zone_mape),
        "WMAPE": nanmean_safe(all_zone_wmape),
        "R2": nanmean_safe(all_zone_r2),
        "n_zones": len(all_zone_mae)
    }


def compute_overall_zone_metrics(zone_level_predictions):
    """Global pooled metrics (giữ lại để tham khảo, KHÔNG dùng so sánh paper)."""
    all_y_true = []
    all_y_pred = []
    for c, pred in zone_level_predictions.items():
        all_y_true.append(pred["y_test"].reshape(-1))
        all_y_pred.append(pred["y_pred"].reshape(-1))
    all_y_true = np.concatenate(all_y_true)
    all_y_pred = np.concatenate(all_y_pred)
    return regression_metrics(all_y_true, all_y_pred)


def fit_predict_cluster_models(
    cluster_ts_pdf: pd.DataFrame,
    lag: int = 12,
    train_ratio: float = 0.8,
    xgb_params: dict = None
):
    if xgb_params is None:
        xgb_params = XGB_CLUSTER_PARAMS
    cluster_models = {}
    cluster_predictions = {}
    cluster_metrics = {}
    cluster_test_indices = {}

    for col in cluster_ts_pdf.columns:
        s = cluster_ts_pdf[col].values.astype(np.float32)
        X, y = make_lagged_supervised(s, lag=lag)
        X_train, X_test, y_train, y_test = chronological_train_test_split(X, y, train_ratio)

        x_scaler = MinMaxScaler()
        y_scaler = MinMaxScaler()
        X_train_sc = x_scaler.fit_transform(X_train)
        X_test_sc = x_scaler.transform(X_test)
        y_train_sc = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()

        model = XGBRegressor(**xgb_params)
        model.fit(X_train_sc, y_train_sc)

        y_pred_sc = model.predict(X_test_sc)
        y_pred = y_scaler.inverse_transform(y_pred_sc.reshape(-1, 1)).ravel()
        y_pred = np.round(np.clip(y_pred, 0, None))

        metrics = regression_metrics(y_test, y_pred)
        cluster_models[col] = {"model": model, "x_scaler": x_scaler, "y_scaler": y_scaler}
        cluster_predictions[col] = {"y_test": y_test, "y_pred": y_pred}
        cluster_metrics[col] = metrics

        all_target_times = cluster_ts_pdf.index[lag:]
        split_idx = int(len(all_target_times) * train_ratio)
        cluster_test_indices[col] = all_target_times[split_idx:]

    return cluster_models, cluster_predictions, cluster_metrics, cluster_test_indices

In [6]:
# ============================================================
# DISAGGREGATION V2 — XGBoost
# Features = [cluster_history] + [zone_history] (cải tiến)
# ============================================================

def build_disaggregation_train_dataset_v2(
    cluster_series: pd.Series,
    zone_ts_pdf: pd.DataFrame,
    zones_in_cluster,
    disagg_lag: int = 12,
    train_ratio: float = 0.8
):
    cluster_values = cluster_series.values.astype(np.float32)
    zone_values = zone_ts_pdf[zones_in_cluster].values.astype(np.float32)
    time_index = cluster_series.index

    X_all, Y_all, T_all = [], [], []
    for t in range(disagg_lag, len(cluster_values)):
        x_cluster = cluster_values[t - disagg_lag:t][::-1]
        x_zones = zone_values[t - disagg_lag:t][::-1].flatten()
        x = np.concatenate([x_cluster, x_zones])
        y = zone_values[t]
        X_all.append(x)
        Y_all.append(y)
        T_all.append(time_index[t])

    X_all = np.asarray(X_all, dtype=np.float32)
    Y_all = np.asarray(Y_all, dtype=np.float32)
    T_all = np.asarray(T_all)

    split_idx = int(len(X_all) * train_ratio)
    return (X_all[:split_idx], X_all[split_idx:],
            Y_all[:split_idx], Y_all[split_idx:], T_all[split_idx:])


def replace_test_inputs_with_predicted_cluster_history_v2(
    full_cluster_actual: pd.Series,
    cluster_test_pred: np.ndarray,
    zone_ts_pdf: pd.DataFrame,
    zones_in_cluster,
    disagg_lag: int,
    train_ratio: float,
    cluster_lag: int
):
    actual = full_cluster_actual.values.astype(np.float32)
    zone_values = zone_ts_pdf[zones_in_cluster].values.astype(np.float32)
    time_index = full_cluster_actual.index
    n_total = len(actual)

    n_targets = n_total - cluster_lag
    split_idx_targets = int(n_targets * train_ratio)
    pred_start_idx = cluster_lag + split_idx_targets

    stitched = actual.copy()
    n_replace = min(len(cluster_test_pred), n_total - pred_start_idx)
    stitched[pred_start_idx:pred_start_idx + n_replace] = cluster_test_pred[:n_replace]

    X_test, T_test = [], []
    test_start_idx = max(pred_start_idx, disagg_lag)
    test_end_idx = pred_start_idx + n_replace

    for t in range(test_start_idx, test_end_idx):
        x_cluster = stitched[t - disagg_lag:t][::-1]
        x_zones = zone_values[t - disagg_lag:t][::-1].flatten()
        x = np.concatenate([x_cluster, x_zones])
        X_test.append(x)
        T_test.append(time_index[t])

    return np.asarray(X_test, dtype=np.float32), np.asarray(T_test)


def fit_disaggregation_models_v2(
    zone_ts_pdf: pd.DataFrame,
    cluster_ts_pdf: pd.DataFrame,
    cluster_map: pd.Series,
    cluster_predictions: dict,
    disagg_lag: int = 12,
    cluster_lag: int = 12,
    train_ratio: float = 0.8,
    xgb_params: dict = None
):
    if xgb_params is None:
        xgb_params = XGB_DISAGG_PARAMS

    disagg_models = {}
    zone_level_predictions = {}
    zone_level_metrics = {}

    for c in sorted(cluster_map.unique()):
        cluster_col = f"cluster_{c}"
        zones = cluster_map[cluster_map == c].index.tolist()
        cluster_series = cluster_ts_pdf[cluster_col]
        n_zones = len(zones)
        n_features = disagg_lag * (1 + n_zones)
        print(f"  cluster_{c}: {n_zones} zones, {n_features} features")

        X_train, _, Y_train, _, _ = build_disaggregation_train_dataset_v2(
            cluster_series=cluster_series,
            zone_ts_pdf=zone_ts_pdf,
            zones_in_cluster=zones,
            disagg_lag=disagg_lag,
            train_ratio=train_ratio
        )

        y_pred_cluster = cluster_predictions[cluster_col]["y_pred"]

        X_test, T_test_pred = replace_test_inputs_with_predicted_cluster_history_v2(
            full_cluster_actual=cluster_series,
            cluster_test_pred=np.asarray(y_pred_cluster, dtype=np.float32),
            zone_ts_pdf=zone_ts_pdf,
            zones_in_cluster=zones,
            disagg_lag=disagg_lag,
            train_ratio=train_ratio,
            cluster_lag=cluster_lag
        )

        test_zone_df = zone_ts_pdf.loc[T_test_pred, zones]
        Y_test = test_zone_df.values.astype(np.float32)

        x_scaler = MinMaxScaler()
        y_scaler = MinMaxScaler()
        X_train_sc = x_scaler.fit_transform(X_train)
        X_test_sc = x_scaler.transform(X_test)
        Y_train_sc = y_scaler.fit_transform(Y_train)

        base_model = XGBRegressor(**xgb_params)
        model = MultiOutputRegressor(base_model, n_jobs=1)
        model.fit(X_train_sc, Y_train_sc)

        Y_pred_sc = model.predict(X_test_sc)
        Y_pred = y_scaler.inverse_transform(Y_pred_sc)
        Y_pred = np.round(np.clip(Y_pred, 0, None))

        per_zone_metrics = {}
        for i, z in enumerate(zones):
            per_zone_metrics[z] = regression_metrics(Y_test[:, i], Y_pred[:, i])

        avg_metrics = {
            "MAE": nanmean_safe([m["MAE"] for m in per_zone_metrics.values()]),
            "RMSE": nanmean_safe([m["RMSE"] for m in per_zone_metrics.values()]),
            "MAPE": nanmean_safe([m["MAPE"] for m in per_zone_metrics.values()]),
            "WMAPE": nanmean_safe([m["WMAPE"] for m in per_zone_metrics.values()]),
            "R2": nanmean_safe([m["R2"] for m in per_zone_metrics.values()]),
        }

        disagg_models[c] = {"model": model, "x_scaler": x_scaler,
                            "y_scaler": y_scaler, "zones": zones}
        zone_level_predictions[c] = {"times": T_test_pred, "zones": zones,
                                     "y_test": Y_test, "y_pred": Y_pred}
        zone_level_metrics[c] = {"avg_metrics": avg_metrics,
                                 "per_zone_metrics": per_zone_metrics}

    return disagg_models, zone_level_predictions, zone_level_metrics

In [7]:
# ============================================================
# PIPELINE V4 — XGBoost
# ============================================================
import time as _time
_pipeline_start = _time.time()

print("Step 1/8: preprocess pickup counts in Spark...")
pickup_counts_sdf = preprocess_pickup_counts_spark(df_input).persist()
print("pickup_counts rows:", pickup_counts_sdf.count())
print("distinct zones:", pickup_counts_sdf.select("zone_id").distinct().count())
print("distinct slots:", pickup_counts_sdf.select("slot_ts").distinct().count())

print("Step 2/8: find active zones in Spark...")
active_zones = get_active_zones_spark(
    pickup_counts_sdf,
    min_mean_pickups_per_slot=MIN_MEAN_PICKUPS_PER_SLOT
)
print("Number of active zones:", len(active_zones))
print("Some active zones:", active_zones[:10])

print("Step 3/8: build dense zone-slot matrix in Spark, then convert to pandas...")
zone_ts_pdf = build_dense_zone_slot_matrix_spark(
    pickup_counts_sdf=pickup_counts_sdf,
    active_zones=active_zones,
    slot_minutes=SLOT_MINUTES
)
print("Dense matrix shape:", zone_ts_pdf.shape)

print("Step 4/8: compute correlation and dissimilarity...")
corr_df, dist_df = compute_correlation_dissimilarity(zone_ts_pdf)

print("Step 5/8: cluster zones (NO singleton removal — đúng theo paper)...")
cluster_map, kernel_df, embedding = cluster_zones_kernel_pca_kmeans(
    dist_df=dist_df,
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE
)
print(cluster_map.value_counts().sort_index())
print(f"Total zones in clusters: {len(cluster_map)}")

print("\n=== CLUSTER DISTRIBUTION ===")
for c_id in sorted(cluster_map.unique()):
    zones = cluster_map[cluster_map == c_id].index.tolist()
    mean_demand = zone_ts_pdf[zones].mean().mean()
    std_demand = zone_ts_pdf[zones].std().mean()
    print(f"Cluster {c_id}: {len(zones)} zones = {zones}, "
          f"mean_demand={mean_demand:.1f}, std={std_demand:.1f}")

print("\nStep 6/8: aggregate cluster demand...")
cluster_ts_pdf = aggregate_cluster_demand(zone_ts_pdf, cluster_map)
print("Cluster demand shape:", cluster_ts_pdf.shape)

# Giải phóng Spark
try: pickup_counts_sdf.unpersist()
except: pass
try: spark.catalog.clearCache()
except: pass
try: spark.stop()
except: pass

print("Step 7/8: cluster-level forecasting (XGBoost)...")
cluster_models, cluster_predictions, cluster_metrics, cluster_test_indices = fit_predict_cluster_models(
    cluster_ts_pdf=cluster_ts_pdf,
    lag=CLUSTER_LAG,
    train_ratio=TRAIN_RATIO,
    xgb_params=XGB_CLUSTER_PARAMS
)

print("\nCluster-level metrics (XGBoost):")
for k, v in cluster_metrics.items():
    print(k, v)

print("\nStep 8/8: disaggregation to zone-level V2 (XGBoost)...")
disagg_models, zone_level_predictions, zone_level_metrics = fit_disaggregation_models_v2(
    zone_ts_pdf=zone_ts_pdf,
    cluster_ts_pdf=cluster_ts_pdf,
    cluster_map=cluster_map,
    cluster_predictions=cluster_predictions,
    disagg_lag=DISAGG_LAG,
    cluster_lag=CLUSTER_LAG,
    train_ratio=TRAIN_RATIO,
    xgb_params=XGB_DISAGG_PARAMS
)

_pipeline_end = _time.time()

print("\nZone-level average metrics by cluster (XGBoost):")
for c, res in zone_level_metrics.items():
    print(f"cluster_{c}:", res["avg_metrics"])

# ============================================================
# METRICS ĐÚNG THEO PAPER: per-zone rồi average
# ============================================================
paper_metrics = compute_paper_zone_averaged_metrics(zone_level_predictions)
print(f"\n============================================================")
print(f"ZONE-LEVEL METRICS ĐÚNG THEO PAPER (XGBoost)")
print(f"(Tính metrics từng zone, rồi average across {paper_metrics['n_zones']} zones)")
print(f"============================================================")
print(f"  RMSE : {paper_metrics['RMSE']:.2f}")
print(f"  MAE  : {paper_metrics['MAE']:.2f}")
print(f"  MAPE : {paper_metrics['MAPE']:.2f}%")
print(f"  WMAPE: {paper_metrics['WMAPE']:.2f}%")
print(f"  R²   : {paper_metrics['R2']:.4f}")
print(f"  Training time: {_pipeline_end - _pipeline_start:.1f}s")
print(f"============================================================")

# Tham khảo: global pooled metrics (KHÔNG dùng để so sánh paper)
overall_zone_metrics = compute_overall_zone_metrics(zone_level_predictions)
print("\n[Tham khảo] Global pooled metrics (KHÔNG so sánh paper):")
print(overall_zone_metrics)

Step 1/8: preprocess pickup counts in Spark...


pickup_counts rows: 1936236
distinct zones: 264
distinct slots: 17519
Step 2/8: find active zones in Spark...
Number of active zones: 57
Some active zones: [4, 13, 24, 41, 42, 43, 45, 48, 50, 68]
Step 3/8: build dense zone-slot matrix in Spark, then convert to pandas...


26/05/08 07:41:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Dense matrix shape: (17520, 57)
Step 4/8: compute correlation and dissimilarity...
Step 5/8: cluster zones (NO singleton removal — đúng theo paper)...
cluster_id
0     2
1    11
2     7
3     6
4     4
5    11
6    15
7     1
Name: count, dtype: int64
Total zones in clusters: 57

=== CLUSTER DISTRIBUTION ===
Cluster 0: 2 zones = [132, 138], mean_demand=155.9, std=103.9
Cluster 1: 11 zones = [13, 50, 100, 137, 141, 170, 229, 233, 237, 239, 263], mean_demand=120.0, std=71.2
Cluster 2: 7 zones = [45, 48, 68, 125, 209, 211, 230], mean_demand=89.6, std=46.6
Cluster 3: 6 zones = [4, 79, 114, 144, 158, 249], mean_demand=80.2, std=57.5
Cluster 4: 4 zones = [41, 42, 74, 224], mean_demand=17.3, std=10.7
Cluster 5: 11 zones = [24, 43, 75, 140, 143, 151, 166, 186, 236, 238, 262], mean_demand=90.8, std=60.2
Cluster 6: 15 zones = [87, 88, 90, 107, 113, 142, 161, 162, 163, 164, 231, 234, 246, 261, 264], mean_demand=120.1, std=72.2
Cluster 7: 1 zones = [148], mean_demand=78.2, std=96.8

Step 6/8: aggr